## generate data

In [5]:
import jax.numpy as jnp
import jax
import numpy as np

chars = "0123456789+= "
EOS = '<|EOS|>'
ctoi = {chars[i]:i for i in range(len(chars))}
ctoi[EOS] = 13
itoc = {i:chars[i] for i in range(len(chars))}
itoc[13] = EOS

MAX_PROMPT_LEN = 17
MAX_OPERAND = 999
seed = 42
b = 10

"""
Current implementation is incredibly slow because all the jnp opertions require dispatch from cpu to accelerator and since it's nt jit, it's all sequential.
Additionally all the batch logic is entirely sequential with accelerator work since it queues up to the accelerator and can't be done in advance on cpu.

Fix: rewrite entirely in np and only do one host->device copy at the end when constructing a jax array.
"""
def generate_batch(b):
    out = []
    loss_mask = []
    for i in range(b):
        operands = np.random.randint(0, high=MAX_OPERAND, size=(2,))
        prompt = f"{operands[0]} + {operands[1]} = "
        prompt_size = len(prompt)
        ans = f"{operands[0] + operands[1]}"
        input = "".join([prompt, ans])
        # print('input:')
        # print(input)
        tokenized_prompt = np.array([ctoi[ch] for ch in input] + [13], dtype=np.int32)
        non_padded_size = tokenized_prompt.shape[0]
        padded_tokenized_input = np.pad(tokenized_prompt, (0, MAX_PROMPT_LEN - tokenized_prompt.shape[0]), constant_values=12)
        # print('padded input')
        # print("".join([itoc[i] for i in padded_tokenized_input.tolist()]))
        inds = np.arange(MAX_PROMPT_LEN)
        # represents what logit tokens will be supervised
        input_loss_mask = np.logical_or(inds >= non_padded_size - 1, inds < prompt_size - 1)[:-1]
        # print('input loss mask:')
        # print(input_loss_mask)
        out.append(padded_tokenized_input)
        loss_mask.append(input_loss_mask)

    batch = np.vstack(out)
    loss_mask = np.vstack(loss_mask)

    return jnp.array(batch, dtype=jnp.int8), jnp.array(loss_mask, dtype=jnp.bool)


batch, loss_mask = generate_batch(2)
# print(loss_mask)
# print('\n\n')
# b = ["".join([itoc[int(i)] for i in b]) for b in batch]
# for i in range(len(b)):
#     print(b[i])
#     print(loss_mask[i])

## model definition

prenorm + rope + swiglu with weight tying for the embed / unembed matrix

use vanilla mha since using grouped mqa or mqa is unnecessary for the scale of this exercise and we aren't doing kv cached inference.

config:

D=256
F=4*256
L=10
N=4
=> ~10.5M params

In [2]:
from dataclasses import dataclass
from flax import nnx

@dataclass
class MoEConfig:
  experts_per_token: int
  experts: int

@dataclass
class ModelConfig:
  hidden_size: int
  ffw_size: int
  layers: int
  attn_heads: int
  vocab_size: int
  moe_config: MoEConfig | None

  def size(self):
      attn_params = 4*self.hidden_size**2 + self.hidden_size
      mlp_params = 3*self.hidden_size*self.ffw_size + self.hidden_size + self.ffw_size
      return self.layers*(attn_params + mlp_params) + self.vocab_size*self.hidden_size


class RMSNorm(nnx.Module):
  def __init__(self, config):
    hidden_size = config.hidden_size
    self.gamma = nnx.Param(jnp.ones((hidden_size,)))

  def __call__(self, x):
    inv_norm = jax.lax.rsqrt(jnp.mean(x**2, axis=-1) + 1e-6)[..., None]
    return self.gamma * x * inv_norm


class RoPE(nnx.Module):
  def _create_inv_freqs(self, base, head_size):
    assert head_size % 2 == 0
    inds = jnp.arange(head_size // 2)
    return base ** (-2 * inds / head_size)

  def _create_cos_sin(self, T, inv_freqs):
    # [1, T, 1, 1]
    positions = jnp.arange(T)[None, :, None, None]
    # [1, 1, 1, H]
    dupl_inv_freqs = jnp.concat([inv_freqs, inv_freqs])[None, None, None, :]
    # [1, T, 1, H]
    freqs = positions * dupl_inv_freqs
    return jnp.cos(freqs), jnp.sin(freqs)

  def _rotate_half(self, x):
    # x[B, T, N, H]
    H = x.shape[-1]
    even = jax.lax.dynamic_slice_in_dim(x, 0, H//2, axis=-1)
    odd = jax.lax.dynamic_slice_in_dim(x, H//2, H//2, axis=-1)
    return jnp.concat([-odd, even], axis=-1)

  def __init__(self, config, base=10_000):
    """

    for each pair of features in input

    x = [[x0],
        [x1]]

    apply rotation matrix:

    R = [[cos0, -sin0]
        [sin0, cos0]]

    Rx =>

    x0' = x0*cos0 - x1*sin0
    x1' = x1*cos0 + x0*sin0

    where 0 = 2*pi*f_j*t

    and f_j = base ** (-2i/d) where i is the index of the pair

    (at i = 0, we have frequency 1 (fast recurrences))
    (at i = d/2, we have frequency base (slow long term))

    pretend top half of x are "even" indices of x
    and bottom half of x are "odd" indices of x
    this doesn't matter since the score doesn't change
    and is just a permutation in projection weights


    x_even' = x_even*cos0 - x_odd*sin0
    x_odd' = x_odd*cos0 + x_even*sin0

    x is split so that top half is even and bottom half is odd

    then this essentially becomes x = x * cos + rotate_half(x) * sin
    where rotate half takes the odd and stacks on top and negates

    note the d dimension of cos and sin must follow this half convention as well
    """
    head_size = config.hidden_size // config.attn_heads
    self.base = base
    self.inv_freqs = self._create_inv_freqs(base, head_size)

  def __call__(self, x):
    """
    pretend top half of x are "even" indices of x
    and bottom half of x are "odd" indices of x
    this doesn't matter since the score doesn't change
    and is just a permutation in projection weights
    """

    B, T, N, H = x.shape
    cos, sin = self._create_cos_sin(T, self.inv_freqs)
    return x * cos + self._rotate_half(x) * sin


class MHA(nnx.Module):
  def __init__(self, config, *, rngs):
    self.config = config
    hidden_size = config.hidden_size
    self.pre_norm = RMSNorm(config)
    self.rope = RoPE(config)
    self.qkv_proj = nnx.Param(0.02 * jax.random.normal(rngs.params(), (hidden_size, 3 * hidden_size)))
    self.o_proj = nnx.Param(0.02 * jax.random.normal(rngs.params(), (hidden_size, hidden_size)))

  def _sdpa(self, q, k, v):
    """
    q[B, T, N, H]
    k[B, S, N, H]
    v[B, S, N, H]
    """
    B, T, N, H = q.shape
    S = k.shape[1]
    logits = jnp.einsum('btnh,bsnh->btsn', q, k)
    scaled_logits = (H ** -0.5) * logits
    # logits[B, T, S, N]
    # mask[1, T, S, 1]
    # non mask token positions are True
    mask = jnp.tril(jnp.ones((T, S), dtype=jnp.bool))[None, :, :, None]
    # [B, T, S, N]
    masked_scaled_logits = jnp.where(mask, scaled_logits, -jnp.inf)
    weights = jax.nn.softmax(masked_scaled_logits, axis=2)
    attn_out = jnp.einsum('btsn,bsnh->btnh', weights, v)
    return attn_out

  def __call__(self, x):
    # x = [B, T, D]
    B, T, D = x.shape
    N = self.config.attn_heads
    H = D//N
    t = self.pre_norm(x)
    qkv = jnp.einsum('btd,df->btf', t, self.qkv_proj)
    # q[B, T, D]
    q = jax.lax.dynamic_slice_in_dim(qkv, 0, D, axis=-1).reshape((B, T, N, H))
    q = self.rope(q)
    k = jax.lax.dynamic_slice_in_dim(qkv, D, D, axis=-1).reshape((B, T, N, H))
    k = self.rope(k)
    v = jax.lax.dynamic_slice_in_dim(qkv, 2*D, D, axis=-1).reshape((B, T, N, H))
    # [B, T, N, H] -> [B, T, D]
    attn_out = self._sdpa(q, k, v).reshape((B, T, D))
    out = jnp.einsum('btd,df->btf', attn_out, self.o_proj)
    return x + out


# first do dense then incorporate sparse moe
class MLP(nnx.Module):
  def __init__(self, config, *, rngs):
    self.config = config
    hidden_size = config.hidden_size
    ffw_size = config.ffw_size
    moe_config = config.moe_config
    self.pre_norm = RMSNorm(config)
    if moe_config:
      E, k = moe_config.experts, moe_config.experts_per_token
      self.router = nnx.Param(0.02 * jax.random.normal(rngs.params(), (hidden_size, E)))
      self.glu_proj = nnx.Param(0.02 * jax.random.normal(rngs.params(), (E, hidden_size, 2*ffw_size)))
      self.down_proj = nnx.Param(0.02 * jax.random.normal(rngs.params(), (E, ffw_size, hidden_size)))
    else:
      self.glu_proj = nnx.Param(0.02 * jax.random.normal(rngs.params(), (hidden_size, 2*ffw_size)))
      self.down_proj = nnx.Param(0.02 * jax.random.normal(rngs.params(), (ffw_size, hidden_size)))
    self.beta = nnx.Param(jnp.ones(ffw_size))



  def __call__(self, x):
    moe_config = self.config.moe_config
    # x[B, T, D]
    # [D, 2F]
    ffw_size = self.config.ffw_size
    t = self.pre_norm(x)

    """
    MoE:
    one router for all 3 proj matrices in matrix
    router: [D, E], softmax and do weighted sum
    glu: [E, D, 2F]
    down_proj: [E, F, D]

    get top k experts for each token:
    [B, T, k]

    flatten router experts into [B*T, k]

    create inds of size [B*T, k] (repeated over k axis)
    flatten into [B*T*k]
    flatten router experts into [B*T*k]
    sort router experts into [B*T*k]
    create group sizes
    create flattened sortened input of [B*T*k, D]
    use ragged dot
    unsort input into [B*T*k, D]
    reshape to [B, T, k, D]
    multiply by expert weights and sum

    flatten input to [B*T, k]

    """


    if moe_config:
      k, E = moe_config.experts_per_token, moe_config.experts
      # [B, T, E]
      router_logits = jnp.einsum('btd,de->bte', t, self.router)
      # [B, T, E]
      router_weights = jax.nn.softmax(router_logits, axis=-1)
      # [B, T, k] [B, T, k]
      top_k_weights, top_k_experts = jax.lax.top_k(router_weights, k)

      B, T, D = t.shape
      # [B * T, D]
      flattened_x = t.reshape((B*T, D))
      # [B * T, k]
      flattened_x_inds_padded = jnp.repeat(jnp.arange(B*T)[:, None], k, axis=-1)
      # [B*T*k]
      flattened_x_inds_padded = flattened_x_inds_padded.reshape((B*T*k,))
      # [B*T*k]
      flattened_routed_experts = top_k_experts.reshape((B*T*k,))
      # [B*T*k]
      # inds has indices from 0 to B*T*k since it returns sorted inds
      sorted_routed_experts_inds = jnp.argsort(flattened_routed_experts)
      # [B*T*k]
      # sorted_routed_experts = flattened_routed_experts[sorted_routed_experts_inds]
      # [E]
      sizes = jnp.bincount(flattened_routed_experts, minlength=E, length=E)


      # [B*T*k]
      # sorted x inds relative to the experts arrays sorted by expert (which token does which expert route correspond to)
      sorted_flattened_x_inds = flattened_x_inds_padded[sorted_routed_experts_inds]
      # [B*T*k, D]
      # x sorted by expert
      sorted_flat_x = flattened_x[sorted_flattened_x_inds]
      # x[B*T*k, D]
      # W[E, D, 2F]
      # => [B*T*k, 2F]
      glu = jax.lax.ragged_dot(sorted_flat_x, self.glu_proj.value, sizes)
      # [B*T*k, F]
      f1 = jax.lax.dynamic_slice_in_dim(glu, 0, ffw_size, axis=-1)
      # [B*T*k, F]
      f2 = jax.lax.dynamic_slice_in_dim(glu, ffw_size, ffw_size, axis=-1)
      # [B*T*k, F]
      swiglu_out = f1 * f2 * jax.nn.sigmoid(self.beta * f2)
      # [B*T*k, D]
      # each token is still in sorted order
      out = jax.lax.ragged_dot(swiglu_out, self.down_proj.value, sizes)
      # [B*T*k, D]
      out = out[jnp.argsort(sorted_routed_experts_inds)]
      # [B, T, k, D]
      out = out.reshape((B, T, k, D))
      # w[B, T, k] -> [B, T, k, 1]
      # out[B, T, k, D]
      # => [B, T, k, D] => [B, T, D]
      out = jnp.sum(top_k_weights[..., None] * out, axis=2)
    else:
      glu = jnp.einsum('btd,df->btf', t, self.glu_proj)
      # [B, T, F]
      f1 = jax.lax.dynamic_slice_in_dim(glu, 0, ffw_size, axis=-1)
      # [B, T, F]
      f2 = jax.lax.dynamic_slice_in_dim(glu, ffw_size, ffw_size, axis=-1)
      t = f1 * f2 * jax.nn.sigmoid(self.beta * f2)
      out = jnp.einsum('btf,fd->btd', t, self.down_proj)

    return x + out


class TransformerLayer(nnx.Module):
  def __init__(self, config, *, rngs):
    self.attn_block = MHA(config, rngs=rngs)
    self.mlp_block = MLP(config, rngs=rngs)

  def __call__(self, x):
    t = self.attn_block(x)
    return self.mlp_block(t)


class Transformer(nnx.Module):
  def __init__(self, config, *, rngs):
    hidden_size = config.hidden_size
    vocab_size = config.vocab_size
    self.embed = nnx.Param(0.02 * jax.random.normal(rngs.params(), (vocab_size, hidden_size)))
    self.layers = [TransformerLayer(config, rngs=rngs) for _ in range(config.layers)]

  def __call__(self, x):
    x = self.embed[x]
    for l in self.layers:
      x = l(x)
    return jnp.einsum('btd,vd->btv', x, self.embed)


In [5]:
# moe mlp test
# TEST

# END TEST

import numpy as np


class NaiveMoEMLP(nnx.Module):
    def __init__(self, config, pre_norm, router, glu_proj, down_proj, beta):
      self.config = config
      self.pre_norm = pre_norm
      self.router = router
      self.glu_proj = glu_proj
      self.down_proj = down_proj
      self.beta = beta

    def __call__(self, x):
        """
        router: [D, E]
        glu_proj: [E, D, 2F]
        down_proj: [E, F, D]
        beta: [F]

        x: [B, T, D]
        """
        k = self.config.moe_config.experts_per_token
        x_n = self.pre_norm(x)
        E, F, D = self.down_proj.value.shape
        B, T, D = x.shape
        # [B, T, E]
        router_logits = jnp.einsum('btd,de->bte', x_n, self.router)
        # [B, T, E]
        router_weights = jax.nn.softmax(router_logits, axis=-1)
        # [B, T, k] [B, T, k]
        top_k_weights, top_k_experts = jax.lax.top_k(router_weights, k)

        out = jnp.zeros((B, T, D))
        for b in range(B):
            for t in range(T):
                # [D]
                tok = x_n[b, t]
                # [k]
                top_k = top_k_experts[b, t]
                # [k, D, 2F]
                k_glu_proj = self.glu_proj[top_k]
                # [k, 2F]
                glu = jnp.einsum('d,kdf->kf', tok, k_glu_proj)
                # [k, F]
                f1 = jax.lax.dynamic_slice_in_dim(glu, 0, F, axis=-1)
                # [k, F]
                f2 = jax.lax.dynamic_slice_in_dim(glu, F, F, axis=-1)
                # [k, F]
                tmp = f1 * f2 * jax.nn.sigmoid(self.beta * f2)
                # [k, F, D]
                k_down_proj = self.down_proj[top_k]
                # [k, D]
                tmp = jnp.einsum('kf,kfd->kd', tmp, k_down_proj)
                # [k]
                k_weights = top_k_weights[b, t]
                # [k, 1] * [k, D] = [k, D] => [D]
                tok_out = jnp.sum(k_weights[:, None] * tmp, axis=0)
                out = out.at[b, t].set(tok_out)

        return x + out


@dataclass
class MoEConfig:
  experts_per_token: int
  experts: int

@dataclass
class ModelConfig:
  hidden_size: int
  ffw_size: int
  layers: int
  attn_heads: int
  vocab_size: int
  moe_config: MoEConfig | None


B, T, D, F, k, E = 2, 2, 4, 16, 2, 4

moe_config = MoEConfig(
  experts_per_token=k,
  experts=E
)

config = ModelConfig(
  hidden_size=D,
  ffw_size=F,
  layers=2,
  attn_heads=2,
  vocab_size=3,
  moe_config=moe_config
)

rngs = nnx.Rngs(0)
moe_mlp = MLP(config, rngs=rngs)
moe_naive = NaiveMoEMLP(config, moe_mlp.pre_norm, moe_mlp.router, moe_mlp.glu_proj, moe_mlp.down_proj, moe_mlp.beta)
x = jax.random.normal(rngs.params(), (B, T, D))
moe_out = moe_mlp(x)
moe_baseline = moe_naive(x)

print('moe_out shape:', moe_out.shape)
print('moe_baseline shape:', moe_baseline.shape)

np.testing.assert_array_almost_equal(moe_out, moe_baseline)

KeyboardInterrupt: 

## basic training loop

config:

D=256
F=4*256
L=10
N=4
=> ~10.5M params

In [3]:
import optax
from optax.losses import softmax_cross_entropy_with_integer_labels as softmax_ce_loss

@nnx.jit
def train_step(model, opt, batch, loss_mask):
    def loss_fn(model):
        labels = batch[:, 1:]
        inv_mask = jnp.logical_not(loss_mask).astype(jnp.int32)
        out = model(batch)
        logits = out[:, :-1]
        return jnp.sum(inv_mask * softmax_ce_loss(logits, labels)) / jnp.sum(inv_mask)

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    opt.update(model, grads)
    return loss


@dataclass
class TrainingConfig:
    max_lr: float
    min_lr: float
    steps: int
    warmup_steps: int
    bs: int
    seed: int
    print_every: int = 100

@dataclass
class TrainingResult:
    losses: list[float]

def train_model(model_config, training_config):
    rngs = nnx.Rngs(training_config.seed)
    model = Transformer(model_config, rngs=rngs)
    lr_schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=training_config.max_lr,
        warmup_steps=training_config.warmup_steps,
        decay_steps=training_config.steps - training_config.warmup_steps,
        end_value=training_config.min_lr
    )
    opt = nnx.Optimizer(model, optax.adamw(learning_rate=lr_schedule), wrt=nnx.Param)
    losses = []

    print(f'--- starting run for model of size {model_config.size():1.2e} ---')
    for step in range(training_config.steps):
        batch, loss_mask = generate_batch(training_config.bs)
        loss = train_step(model, opt, batch, loss_mask)
        # append jax future value and force a barrier every print_every steps.
        # by doing this, the host can continue generating batches and asynchronously
        # queue up work to the accelerator.
        losses.append(loss)
        if step % training_config.print_every == 0:
            print(f"step: {step}, loss: {loss}")
            losses = [float(loss) for loss in losses]

    return TrainingResult(losses=losses)


## preliminary runs
run a subset of different model sizes to see at what point their loss hits close to 0 (the problem has been completely solved) to choose appropriate C horizon lengths for Chinchilla laws

In [8]:
import matplotlib.pyplot as plt
import os

# use 300K, 1.2M, and 4.8M model sizes
preliminary_model_configs = [
    # 300K
    ModelConfig(
      hidden_size=64,
      ffw_size=4*64,
      layers=5,
      attn_heads=4,
      vocab_size=14,
      moe_config=None
    ),

    # 1.2M
    ModelConfig(
        hidden_size=128,
        ffw_size=4*128,
        layers=5,
        attn_heads=4,
        vocab_size=14,
        moe_config=None
    ),

    # ~5M
    ModelConfig(
        hidden_size=256,
        ffw_size=4*256,
        layers=5,
        attn_heads=4,
        vocab_size=14,
        moe_config=None
    ),
]

FLOPS_HORIZON = 1e14
def flops_to_training_steps(model_size, flops_horizon, b, t):
    flops_per_step = 6 * model_size * b * t
    return int(flops_horizon // flops_per_step)

training_configs = []
for m in preliminary_model_configs:
    steps = flops_to_training_steps(m.size(), FLOPS_HORIZON, 32, 17)
    training_configs.append(
        TrainingConfig(
            max_lr=1e-3,
            min_lr=1e-4,
            steps=steps,
            warmup_steps=int(0.3*steps),
            bs=32,
            seed=42,
            print_every=1000
        )
    )

for training_config in training_configs:
    print(training_config.steps)

training_results = []
os.makedirs('preliminary', exist_ok=True)
for i in range(len(training_configs)):
    model_config = preliminary_model_configs[i]
    training_config = training_configs[i]
    res = train_model(model_config, training_config)
    training_results.append(res)
    # save results to file
    with open(f'preliminary/{model_config.size():1.2e}.npy', 'wb') as f:
        np.save(f, np.array(res.losses))


for i in range(len(preliminary_model_configs)):
    model_config = preliminary_model_configs[i]
    training_config = training_configs[i]
    res = training_results[i]
    plt.plot(np.arange(training_config.steps), np.array(res.losses), label=f"{config.size():1.2e}")

92700
23274
5831
--- starting run for model of size 3.30e+05 ---
step: 0, loss: 2.6411569118499756
step: 1000, loss: 1.6773478984832764
step: 2000, loss: 1.4065823554992676
step: 3000, loss: 1.2659196853637695
step: 4000, loss: 1.0880236625671387
step: 5000, loss: 1.1065114736557007
step: 6000, loss: 1.1414510011672974


KeyboardInterrupt: 